In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q sae-lens transformer-lens datasets

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 12.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.0/311.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.7/977.7 kB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 26.2 MB/s eta 0:00:00


In [ ]:
import torch
import os

ACT_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/layer6_resid_post (1).pt"

print("Exists:", os.path.exists(ACT_PATH))
print("Size MB:", os.path.getsize(ACT_PATH) / 1024 / 1024)

acts = torch.load(ACT_PATH, map_location="cpu")

print("Shape:", acts.shape)
print("Dtype:", acts.dtype)
print("Mean:", acts.mean().item())
print("Std:", acts.std().item())

Exists: True
Size MB: 408.8365316390991
Shape: torch.Size([139549, 768])
Dtype: torch.float32
Mean: 2.9158291336983666e-09
Std: 1.6805423498153687


In [ ]:
import torch

device = "cuda"

acts_gpu = acts.to(device)

print(acts_gpu.shape)
print(torch.cuda.memory_allocated() / 1024**3, "GB")

torch.Size([139549, 768])
0.39925289154052734 GB


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

device = "cuda"

acts = acts.float()

dataset = TensorDataset(acts)
loader = DataLoader(
    dataset,
    batch_size=1024,
    shuffle=True,
    drop_last=True,
)

print("Samples:", len(dataset))
print("Batches:", len(loader))

Samples: 139549
Batches: 136


In [ ]:
from sae_lens import StandardSAEConfig, StandardSAE

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device="cuda",
)

sae = StandardSAE(cfg).to(device)

print(sae)

StandardSAE(
  (activation_fn): ReLU()
  (hook_sae_input): HookPoint(name='hook_sae_input')
  (hook_sae_acts_pre): HookPoint(name='hook_sae_acts_pre')
  (hook_sae_acts_post): HookPoint(name='hook_sae_acts_post')
  (hook_sae_output): HookPoint(name='hook_sae_output')
  (hook_sae_recons): HookPoint(name='hook_sae_recons')
  (hook_sae_error): HookPoint(name='hook_sae_error')
)


In [ ]:
optimizer = torch.optim.Adam(
    sae.parameters(),
    lr=1e-4,
)

In [ ]:
import torch
from tqdm.auto import tqdm

EPOCHS = 10
L1_COEFF = 1e-4

for epoch in range(EPOCHS):

    sae.train()

    total_loss = 0.0
    total_recon = 0.0
    total_l1 = 0.0

    pbar = tqdm(loader)

    for (batch,) in pbar:

        batch = batch.to(device)

        optimizer.zero_grad()

        # encode
        features = sae.encode(batch)

        # decode
        recon = sae.decode(features)

        recon_loss = torch.mean((recon - batch) ** 2)

        l1_loss = features.abs().mean()

        loss = recon_loss + L1_COEFF * l1_loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_l1 += l1_loss.item()

        pbar.set_description(
            f"loss={loss.item():.6f} "
            f"recon={recon_loss.item():.6f} "
            f"l1={l1_loss.item():.6f}"
        )

    print(
        f"\nEpoch {epoch+1}/{EPOCHS} | "
        f"Loss={total_loss/len(loader):.6f} | "
        f"Recon={total_recon/len(loader):.6f} | "
        f"L1={total_l1/len(loader):.6f}"
    )

  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 1/10 | Loss=54.707023 | Recon=54.706992 | L1=0.311982


  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 2/10 | Loss=6.132016 | Recon=6.131999 | L1=0.165060


  0%|          | 0/136 [00:00<?, ?it/s]

...


Epoch 8/10 | Loss=0.258859 | Recon=0.258851 | L1=0.081830


  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 9/10 | Loss=0.230926 | Recon=0.230918 | L1=0.080248


  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 10/10 | Loss=0.208607 | Recon=0.208599 | L1=0.079405


In [ ]:
import torch

sae.eval()

with torch.no_grad():

    batch = acts[:4096].cuda()

    features = sae.encode(batch)

    active = (features > 0).float().sum(dim=1)

    print("Mean active features:", active.mean().item())
    print("Median active features:", active.median().item())
    print("Max active features:", active.max().item())
    print("Min active features:", active.min().item())

Mean active features: 367.197265625
Median active features: 307.0
Max active features: 2771.0
Min active features: 79.0


In [ ]:
torch.save(
    sae.state_dict(),
    "/content/drive/MyDrive/Mechanistic_Interpretability/base_sae.pt"
)

In [ ]:
import os

print(
    os.path.getsize(
        "/content/drive/MyDrive/Mechanistic_Interpretability/base_sae.pt"
    ) / 1024 / 1024,
    "MB"
)

36.02890682220459 MB


In [ ]:
import torch
from tqdm.auto import tqdm

EPOCHS = 10
L1_COEFF = 1e-2

for epoch in range(EPOCHS):

    sae.train()

    total_loss = 0.0
    total_recon = 0.0
    total_l1 = 0.0

    pbar = tqdm(loader)

    for (batch,) in pbar:

        batch = batch.to(device)

        optimizer.zero_grad()

        # encode
        features = sae.encode(batch)

        # decode
        recon = sae.decode(features)

        recon_loss = torch.mean((recon - batch) ** 2)

        l1_loss = features.abs().mean()

        loss = recon_loss + L1_COEFF * l1_loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_l1 += l1_loss.item()

        pbar.set_description(
            f"loss={loss.item():.6f} "
            f"recon={recon_loss.item():.6f} "
            f"l1={l1_loss.item():.6f}"
        )

    print(
        f"\nEpoch {epoch+1}/{EPOCHS} | "
        f"Loss={total_loss/len(loader):.6f} | "
        f"Recon={total_recon/len(loader):.6f} | "
        f"L1={total_l1/len(loader):.6f}"
    )

  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 1/10 | Loss=0.190303 | Recon=0.189512 | L1=0.079083


  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 2/10 | Loss=0.174025 | Recon=0.173234 | L1=0.079077


  0%|          | 0/136 [00:00<?, ?it/s]

...


Epoch 8/10 | Loss=0.115848 | Recon=0.115037 | L1=0.081099


  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 9/10 | Loss=0.108496 | Recon=0.107671 | L1=0.082430


  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 10/10 | Loss=0.101503 | Recon=0.100662 | L1=0.084037


In [ ]:
import torch

sae.eval()

with torch.no_grad():

    batch = acts[:4096].cuda()

    features = sae.encode(batch)

    active = (features > 0).float().sum(dim=1)

    print("Mean active features:", active.mean().item())
    print("Median active features:", active.median().item())
    print("Max active features:", active.max().item())
    print("Min active features:", active.min().item())

Mean active features: 352.248291015625
Median active features: 309.0
Max active features: 2709.0
Min active features: 204.0


In [ ]:
torch.save(
    sae.state_dict(),
    "/content/drive/MyDrive/Mechanistic_Interpretability/base_sae_final.pt"
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE = "/content/drive/MyDrive/Mechanistic_Interpretability"

folders = [
    "activations/base",
    "activations/finetuned",
    "sae_models/base",
    "sae_models/finetuned",
    "finetuned_models",
    "results",
    "scripts",
]

for f in folders:
    os.makedirs(os.path.join(BASE, f), exist_ok=True)

print("Done")

Done


In [ ]:
import torch

print(torch.cuda.get_device_name(0))
print(
    torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "GB"
)

Tesla T4
14.56317138671875 GB


In [ ]:
!pip install -q transformers datasets accelerate peft

In [ ]:
import transformers
import datasets
import peft
import accelerate

print(transformers.__version__)
print(datasets.__version__)
print(peft.__version__)
print(accelerate.__version__)

5.9.0
4.0.0
0.19.1
1.13.0


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "iamtarun/python_code_instructions_18k_alpaca",
    split="train"
)

print(dataset)
print(dataset.column_names)
print(dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/905 [00:00<?, ?B/s]

data/train-00000-of-00001-8b6e212f3e1ece(…):   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18612 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output', 'prompt'],
    num_rows: 18612
})
['instruction', 'input', 'output', 'prompt']
{'instruction': 'Create a function to calculate the sum of a sequence of integers.', 'input': '[1, 2, 3, 4, 5]', 'output': '# Python code\ndef sum_sequence(sequence):\n  sum = 0\n  for num in sequence:\n    sum += num\n  return sum', 'prompt': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nCreate a function to calculate the sum of a sequence of integers.\n\n### Input:\n[1, 2, 3, 4, 5]\n\n### Output:\n# Python code\ndef sum_sequence(sequence):\n  sum = 0\n  for num in sequence:\n    sum += num\n  return sum'}


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "EleutherAI/pythia-160m"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

<|padding|>


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "iamtarun/python_code_instructions_18k_alpaca",
    split="train"
)

def tokenize_fn(example):
    return tokenizer(
        example["prompt"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=False,
    remove_columns=dataset.column_names,
)

print(tokenized_dataset)
print(tokenized_dataset[0].keys())

Map:   0%|          | 0/18612 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 18612
})
dict_keys(['input_ids', 'attention_mask'])


In [ ]:
from transformers import AutoModelForCausalLM

MODEL_NAME = "EleutherAI/pythia-160m"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print(
    sum(p.numel() for p in model.parameters()) / 1e6,
    "Million parameters"
)

model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

162.322944 Million parameters


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python",

    num_train_epochs=1,

    per_device_train_batch_size=8,

    learning_rate=5e-5,

    weight_decay=0.01,

    logging_steps=50,

    save_steps=500,

    save_total_limit=2,

    report_to="none",
)

In [ ]:
tokenized_dataset = tokenized_dataset.map(
    lambda x: {"labels": x["input_ids"]}
)

print(tokenized_dataset[0].keys())

Map:   0%|          | 0/18612 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

In [ ]:
print(model.config)
print(tokenizer.pad_token)
print(tokenizer.pad_token_id)
print(tokenizer.eos_token)
print(tokenizer.eos_token_id)

GPTNeoXConfig {
  "architectures": [
    "GPTNeoXForCausalLM"
  ],
  "attention_bias": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.1,
  "dtype": "float16",
  "eos_token_id": 0,
  "hidden_act": "gelu",
  "hidden_dropout": 0.0,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 2048,
  "model_type": "gpt_neox",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": null,
  "rope_parameters": {
    "partial_rotary_factor": 0.25,
    "rope_theta": 10000,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.9.0",
  "use_cache": false,
  "use_parallel_residual": true,
  "vocab_size": 50304
}

<|padding|>
1
<|endoftext|>
0


In [ ]:
sample = tokenized_dataset[0]

print(len(sample["input_ids"]))
print(len(sample["labels"]))

print(sample["input_ids"][:20])
print(sample["labels"][:20])

print("Pad count:",
      sum(x == tokenizer.pad_token_id
          for x in sample["labels"]))

256
256
[30003, 310, 271, 9775, 326, 8631, 247, 4836, 15, 19566, 247, 2380, 326, 20420, 29141, 253, 2748, 15, 187, 187]
[30003, 310, 271, 9775, 326, 8631, 247, 4836, 15, 19566, 247, 2380, 326, 20420, 29141, 253, 2748, 15, 187, 187]
Pad count: 164


In [ ]:
import numpy as np

pad_id = tokenizer.pad_token_id

counts = []

for i in range(100):
    labels = tokenized_dataset[i]["labels"]
    counts.append(sum(x == pad_id for x in labels))

print("Average pads:", np.mean(counts))
print("Max pads:", np.max(counts))
print("Min pads:", np.min(counts))

Average pads: 97.59
Max pads: 188
Min pads: 0


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "iamtarun/python_code_instructions_18k_alpaca",
    split="train"
)

def tokenize_fn(example):

    encoded = tokenizer(
        example["prompt"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

    labels = encoded["input_ids"].copy()

    labels = [
        token if token != tokenizer.pad_token_id
        else -100
        for token in labels
    ]

    encoded["labels"] = labels

    return encoded

tokenized_dataset = dataset.map(
    tokenize_fn,
    remove_columns=dataset.column_names,
)

Map:   0%|          | 0/18612 [00:00<?, ? examples/s]

In [ ]:
sample = tokenized_dataset[0]

print(sample["labels"][:30])

print(
    "Ignored labels:",
    sum(x == -100 for x in sample["labels"])
)

[30003, 310, 271, 9775, 326, 8631, 247, 4836, 15, 19566, 247, 2380, 326, 20420, 29141, 253, 2748, 15, 187, 187, 4118, 41959, 27, 187, 9395, 247, 1159, 281, 10173, 253]
Ignored labels: 164


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/pythia-160m"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python",

    num_train_epochs=1,

    per_device_train_batch_size=8,

    learning_rate=2e-5,

    weight_decay=0.01,

    logging_steps=50,

    save_steps=500,

    save_total_limit=2,

    report_to="none",
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

In [ ]:
trainer.train()

{'loss': '711.4', 'grad_norm': '3176', 'learning_rate': '1.958e-05', 'epoch': '0.02149'}
{'loss': '1138', 'grad_norm': 'nan', 'learning_rate': '1.915e-05', 'epoch': '0.04297'}
{'loss': '0', 'grad_norm': 'nan', 'learning_rate': '1.872e-05', 'epoch': '0.06446'}


KeyboardInterrupt: 

In [ ]:
print(model.dtype)

for name, param in model.named_parameters():
    print(name, param.dtype)
    break


torch.float16
gpt_neox.embed_in.weight torch.float16


In [ ]:
print("Max token id:", max(tokenized_dataset[0]["input_ids"]))
print("Min token id:", min(tokenized_dataset[0]["input_ids"]))
print("Vocab size:", model.config.vocab_size)

Max token id: 50276
Min token id: 1
Vocab size: 50304


In [ ]:
import torch

model.eval()

sample = tokenized_dataset[0]

input_ids = torch.tensor([sample["input_ids"]]).cuda()
attention_mask = torch.tensor([sample["attention_mask"]]).cuda()
labels = torch.tensor([sample["labels"]]).cuda()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

print("Loss:", outputs.loss.item())

Loss: nan


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/pythia-160m",
    torch_dtype=torch.float32
).cuda()

print(model.dtype)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

torch.float32


In [ ]:
import torch

model.eval()

sample = tokenized_dataset[0]

input_ids = torch.tensor([sample["input_ids"]]).cuda()
attention_mask = torch.tensor([sample["attention_mask"]]).cuda()
labels = torch.tensor([sample["labels"]]).cuda()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

print(outputs.loss)

tensor(2.2920, device='cuda:0')


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/pythia-160m",
    torch_dtype=torch.float32
).cuda()

print(model.dtype)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

torch.float32


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python",

    num_train_epochs=1,

    per_device_train_batch_size=8,

    learning_rate=2e-5,

    weight_decay=0.01,

    logging_steps=50,

    save_steps=500,

    save_total_limit=2,

    report_to="none",

    fp16=False,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

In [ ]:
trainer.train()

{'loss': '1.299', 'grad_norm': '13.85', 'learning_rate': '1.958e-05', 'epoch': '0.02149'}
{'loss': '1.182', 'grad_norm': '14.65', 'learning_rate': '1.915e-05', 'epoch': '0.04297'}
{'loss': '1.133', 'grad_norm': '11.3', 'learning_rate': '1.872e-05', 'epoch': '0.06446'}
{'loss': '1.116', 'grad_norm': '9.853', 'learning_rate': '1.829e-05', 'epoch': '0.08595'}
{'loss': '1.082', 'grad_norm': '10.44', 'learning_rate': '1.786e-05', 'epoch': '0.1074'}
{'loss': '1.066', 'grad_norm': '10.33', 'learning_rate': '1.743e-05', 'epoch': '0.1289'}
{'loss': '1.094', 'grad_norm': '9.414', 'learning_rate': '1.7e-05', 'epoch': '0.1504'}
{'loss': '1.034', 'grad_norm': '9.009', 'learning_rate': '1.657e-05', 'epoch': '0.1719'}
{'loss': '1.049', 'grad_norm': '9.66', 'learning_rate': '1.614e-05', 'epoch': '0.1934'}
{'loss': '1.056', 'grad_norm': '9.492', 'learning_rate': '1.571e-05', 'epoch': '0.2149'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.06', 'grad_norm': '13.62', 'learning_rate': '1.528e-05', 'epoch': '0.2364'}
{'loss': '0.9742', 'grad_norm': '7.683', 'learning_rate': '1.485e-05', 'epoch': '0.2578'}
{'loss': '0.9952', 'grad_norm': '8.163', 'learning_rate': '1.442e-05', 'epoch': '0.2793'}
{'loss': '0.9882', 'grad_norm': '10.22', 'learning_rate': '1.399e-05', 'epoch': '0.3008'}
{'loss': '0.9461', 'grad_norm': '8.662', 'learning_rate': '1.356e-05', 'epoch': '0.3223'}
{'loss': '0.959', 'grad_norm': '8.654', 'learning_rate': '1.313e-05', 'epoch': '0.3438'}
{'loss': '0.9647', 'grad_norm': '8.269', 'learning_rate': '1.27e-05', 'epoch': '0.3653'}
{'loss': '0.9499', 'grad_norm': '8.378', 'learning_rate': '1.227e-05', 'epoch': '0.3868'}
{'loss': '0.9837', 'grad_norm': '8.753', 'learning_rate': '1.184e-05', 'epoch': '0.4083'}
{'loss': '0.9957', 'grad_norm': '8.298', 'learning_rate': '1.141e-05', 'epoch': '0.4297'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.9812', 'grad_norm': '9.35', 'learning_rate': '1.098e-05', 'epoch': '0.4512'}
{'loss': '0.9702', 'grad_norm': '7.86', 'learning_rate': '1.055e-05', 'epoch': '0.4727'}
{'loss': '0.9439', 'grad_norm': '7.909', 'learning_rate': '1.012e-05', 'epoch': '0.4942'}
{'loss': '0.9536', 'grad_norm': '7.668', 'learning_rate': '9.695e-06', 'epoch': '0.5157'}
{'loss': '0.9617', 'grad_norm': '8.737', 'learning_rate': '9.265e-06', 'epoch': '0.5372'}
{'loss': '0.9447', 'grad_norm': '8.761', 'learning_rate': '8.835e-06', 'epoch': '0.5587'}
{'loss': '0.9128', 'grad_norm': '8.349', 'learning_rate': '8.406e-06', 'epoch': '0.5801'}
{'loss': '0.9437', 'grad_norm': '9.339', 'learning_rate': '7.976e-06', 'epoch': '0.6016'}
{'loss': '0.9301', 'grad_norm': '8.707', 'learning_rate': '7.546e-06', 'epoch': '0.6231'}
{'loss': '0.8933', 'grad_norm': '9.15', 'learning_rate': '7.116e-06', 'epoch': '0.6446'}


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.8705', 'grad_norm': '7.688', 'learning_rate': '2.389e-06', 'epoch': '0.881'}
{'loss': '0.8304', 'grad_norm': '8.692', 'learning_rate': '1.96e-06', 'epoch': '0.9024'}
{'loss': '0.8962', 'grad_norm': '8.444', 'learning_rate': '1.53e-06', 'epoch': '0.9239'}
{'loss': '0.8454', 'grad_norm': '7.455', 'learning_rate': '1.1e-06', 'epoch': '0.9454'}
{'loss': '0.9029', 'grad_norm': '7.166', 'learning_rate': '6.704e-07', 'epoch': '0.9669'}
{'loss': '0.8424', 'grad_norm': '7.636', 'learning_rate': '2.407e-07', 'epoch': '0.9884'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1367', 'train_samples_per_second': '13.61', 'train_steps_per_second': '1.702', 'train_loss': '0.9645', 'epoch': '1'}


TrainOutput(global_step=2327, training_loss=0.9644619761796074, metrics={'train_runtime': 1367.1391, 'train_samples_per_second': 13.614, 'train_steps_per_second': 1.702, 'train_loss': 0.9644619761796074, 'epoch': 1.0})

In [ ]:
trainer.save_model(
    "/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python_final"
)

tokenizer.save_pretrained(
    "/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python_final"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python_final/tokenizer_config.json',
 '/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python_final/tokenizer.json')

In [ ]:
import os

MODEL_DIR = "/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python_final"

print(os.listdir(MODEL_DIR))

['config.json', 'generation_config.json', 'model.safetensors', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
from transformer_lens import HookedTransformer

MODEL_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python_final"

model = HookedTransformer.from_pretrained(
    MODEL_PATH
)

print("Loaded!")

ValueError: /content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python_final not found. Valid official model names (excl aliases): ['01-ai/Yi-34B', '01-ai/Yi-34B-Chat', '01-ai/Yi-6B', '01-ai/Yi-6B-Chat', 'ai-forever/mGPT', 'allenai/OLMo-1B-hf', 'allenai/OLMo-2-0425-1B', 'allenai/OLMo-2-1124-7B', 'allenai/Olmo-3-32B-Think', 'allenai/Olmo-3-7B-Instruct', 'allenai/Olmo-3-7B-Think', 'allenai/Olmo-3.1-32B-Instruct', 'allenai/Olmo-3.1-32B-Think', 'allenai/OLMo-7B-hf', 'allenai/OLMoE-1B-7B-0924', 'ArthurConmy/redwood_attn_2l', 'Baidicoot/Othello-GPT-Transformer-Lens', 'bigcode/santacoder', 'bigscience/bloom-1b1', 'bigscience/bloom-1b7', 'bigscience/bloom-3b', 'bigscience/bloom-560m', 'bigscience/bloom-7b1', 'codellama/CodeLlama-7b-hf', 'codellama/CodeLlama-7b-Instruct-hf', 'codellama/CodeLlama-7b-Python-hf', 'distilgpt2', 'EleutherAI/gpt-j-6B', 'EleutherAI/gpt-neo-1.3B', 'EleutherAI/gpt-neo-125M', 'EleutherAI/gpt-neo-2.7B', 'EleutherAI/gpt-neox-20b', 'EleutherAI/pythia

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q sae-lens transformer-lens datasets

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.0/311.0 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.7/977.7 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 27.1 MB/s eta 0:00:00


In [ ]:
import os

ROOT = "/content/drive/MyDrive"

for root, dirs, files in os.walk(ROOT):
    for f in files:
        if f.endswith(".pt"):
            print(os.path.join(root, f))

/content/drive/MyDrive/Mechanistic_Interpretability/activations/base/layer6_resid_post (1).pt
/content/drive/MyDrive/Mechanistic_Interpretability/activations/base/layer6_resid_post.pt
/content/drive/MyDrive/Mechanistic_Interpretability/activations/finetuned/layer6_resid_post_finetuned.pt
/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/base/base_sae.pt
/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/base/base_sae_final.pt
/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/finetuned/finetuned_sae_final.pt
/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python/checkpoint-2000/optimizer.pt
/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python/checkpoint-2000/scheduler.pt
/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python/checkpoint-2327/optimizer.pt
/content/drive/MyDrive/Mechanistic_Interpretability/finetuned_models/pythia_python/checkpoint-2327/scheduler.pt
/

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

from sae_lens import StandardSAEConfig, StandardSAE

# ============================================================
# CONFIG
# ============================================================

ACT_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/activations/finetuned/layer6_resid_post_finetuned.pt"

SAVE_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/finetuned/finetuned_sae_final.pt"

BATCH_SIZE = 1024
LR = 1e-4

EPOCHS = 10
L1_COEFF = 1e-2

device = "cuda"

# ============================================================
# LOAD ACTIVATIONS
# ============================================================

print("Loading activations...")

acts = torch.load(
    ACT_PATH,
    map_location="cpu"
)

print("Shape:", acts.shape)
print("Mean:", acts.mean())
print("Std:", acts.std())

dataset = TensorDataset(acts)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

# ============================================================
# SAE
# ============================================================

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device="cuda",
)

sae = StandardSAE(cfg).to(device)

print(sae)

# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.Adam(
    sae.parameters(),
    lr=LR
)

# ============================================================
# TRAIN
# ============================================================

for epoch in range(EPOCHS):

    sae.train()

    total_loss = 0.0
    total_recon = 0.0
    total_l1 = 0.0

    pbar = tqdm(loader)

    for (batch,) in pbar:

        batch = batch.to(device)

        optimizer.zero_grad()

        features = sae.encode(batch)

        recon = sae.decode(features)

        recon_loss = torch.mean((recon - batch) ** 2)

        l1_loss = features.abs().mean()

        loss = recon_loss + L1_COEFF * l1_loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_l1 += l1_loss.item()

        pbar.set_description(
            f"loss={loss.item():.6f} "
            f"recon={recon_loss.item():.6f} "
            f"l1={l1_loss.item():.6f}"
        )

    print(
        f"\nEpoch {epoch+1}/{EPOCHS} | "
        f"Loss={total_loss/len(loader):.6f} | "
        f"Recon={total_recon/len(loader):.6f} | "
        f"L1={total_l1/len(loader):.6f}"
    )

# ============================================================
# SPARSITY CHECK
# ============================================================

sae.eval()

with torch.no_grad():

    batch = acts[:4096].to(device)

    features = sae.encode(batch)

    active = (features > 0).float().sum(dim=1)

    print("\nMean active features:", active.mean().item())
    print("Median active features:", active.median().item())
    print("Max active features:", active.max().item())
    print("Min active features:", active.min().item())

# ============================================================
# SAVE
# ============================================================

torch.save(
    sae.state_dict(),
    SAVE_PATH
)

print(f"\nSaved SAE -> {SAVE_PATH}")

Loading activations...
Shape: torch.Size([139549, 768])
Mean: tensor(-0.0119)
Std: tensor(1.7312)
StandardSAE(
  (activation_fn): ReLU()
  (hook_sae_input): HookPoint(name='hook_sae_input')
  (hook_sae_acts_pre): HookPoint(name='hook_sae_acts_pre')
  (hook_sae_acts_post): HookPoint(name='hook_sae_acts_post')
  (hook_sae_output): HookPoint(name='hook_sae_output')
  (hook_sae_recons): HookPoint(name='hook_sae_recons')
  (hook_sae_error): HookPoint(name='hook_sae_error')
)


  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 1/10 | Loss=52.497803 | Recon=52.494554 | L1=0.324951


  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 2/10 | Loss=5.401720 | Recon=5.399869 | L1=0.185021


...


Epoch 8/10 | Loss=0.229148 | Recon=0.228228 | L1=0.091993


  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 9/10 | Loss=0.208781 | Recon=0.207874 | L1=0.090763


  0%|          | 0/136 [00:00<?, ?it/s]


Epoch 10/10 | Loss=0.193183 | Recon=0.192278 | L1=0.090434

Mean active features: 360.984619140625
Median active features: 316.0
Max active features: 2701.0
Min active features: 145.0

Saved SAE -> /content/drive/MyDrive/Mechanistic_Interpretability/sae_models/finetuned/finetuned_sae_final.pt


In [ ]:
import torch
from sae_lens import StandardSAEConfig, StandardSAE

device = "cpu"

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device=device,
)

base_sae = StandardSAE(cfg)
base_sae.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/base/base_sae_final.pt",
        map_location=device
    )
)

finetuned_sae = StandardSAE(cfg)
finetuned_sae.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/finetuned/finetuned_sae_final.pt",
        map_location=device
    )
)

base_dec = base_sae.W_dec.detach()
finetuned_dec = finetuned_sae.W_dec.detach()

diff = (base_dec - finetuned_dec).norm()

print("Decoder difference:", diff.item())

Decoder difference: 152.46279907226562


In [ ]:
import torch.nn.functional as F

cos = F.cosine_similarity(
    base_dec.flatten().unsqueeze(0),
    finetuned_dec.flatten().unsqueeze(0)
)

print("Global cosine similarity:", cos.item())

Global cosine similarity: 0.00013516651233658195


In [ ]:
!pip install -q sae-lens transformer-lens datasets

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 10.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.0/311.0 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.7/977.7 kB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 25.7 MB/s eta 0:00:00


In [ ]:
print(base_sae.W_dec.shape)
print(finetuned_sae.W_dec.shape)

print(base_sae.W_dec.mean().item())
print(finetuned_sae.W_dec.mean().item())

print(base_sae.W_dec.std().item())
print(finetuned_sae.W_dec.std().item())

torch.Size([6144, 768])
torch.Size([6144, 768])
-1.2854304259235505e-05
2.6127940145670436e-05
0.04938356205821037
0.0498930960893631


In [ ]:
from sae_lens import StandardSAEConfig, StandardSAE

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device="cuda",
)

sae = StandardSAE(cfg).to(device)

print(sae)

StandardSAE(
  (activation_fn): ReLU()
  (hook_sae_input): HookPoint(name='hook_sae_input')
  (hook_sae_acts_pre): HookPoint(name='hook_sae_acts_pre')
  (hook_sae_acts_post): HookPoint(name='hook_sae_acts_post')
  (hook_sae_output): HookPoint(name='hook_sae_output')
  (hook_sae_recons): HookPoint(name='hook_sae_recons')
  (hook_sae_error): HookPoint(name='hook_sae_error')
)


In [ ]:
import torch

base_acts = torch.load(
    "/content/drive/MyDrive/Mechanistic_Interpretability/activations/base/layer6_resid_post (1).pt",
    map_location="cpu"
)

ft_acts = torch.load(
    "/content/drive/MyDrive/Mechanistic_Interpretability/activations/finetuned/layer6_resid_post_finetuned.pt",
    map_location="cpu"
)

def mse(sae, acts):

    sae.eval()

    with torch.no_grad():

        x = acts[:10000]

        feats = sae.encode(x)

        recon = sae.decode(feats)

        return ((recon - x) ** 2).mean().item()

print("Base SAE on Base Acts:", mse(base_sae, base_acts))
print("Base SAE on FT Acts:", mse(base_sae, ft_acts))

print("FT SAE on Base Acts:", mse(finetuned_sae, base_acts))
print("FT SAE on FT Acts:", mse(finetuned_sae, ft_acts))

Base SAE on Base Acts: 0.10266569256782532
Base SAE on FT Acts: 0.20533700287342072
FT SAE on Base Acts: 0.5928015112876892
FT SAE on FT Acts: 0.19146378338336945


In [ ]:
import torch

BASE_SAE_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/base/base_sae_final.pt"
FT_SAE_PATH   = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/finetuned/finetuned_sae_final.pt"

base_ckpt = torch.load(BASE_SAE_PATH, map_location="cpu")
ft_ckpt   = torch.load(FT_SAE_PATH, map_location="cpu")

print(type(base_ckpt))

if isinstance(base_ckpt, dict):
    print("\nBASE KEYS:")
    print(base_ckpt.keys())

if isinstance(ft_ckpt, dict):
    print("\nFT KEYS:")
    print(ft_ckpt.keys())

<class 'collections.OrderedDict'>

BASE KEYS:
odict_keys(['b_dec', 'W_dec', 'W_enc', 'b_enc'])

FT KEYS:
odict_keys(['b_dec', 'W_dec', 'W_enc', 'b_enc'])


In [ ]:
print(base_ckpt["W_dec"].shape)
print(ft_ckpt["W_dec"].shape)

torch.Size([6144, 768])
torch.Size([6144, 768])


In [ ]:
import torch
import numpy as np

BASE_SAE_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/base/base_sae_final.pt"
FT_SAE_PATH   = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/finetuned/finetuned_sae_final.pt"

print("Loading checkpoints...")

base_ckpt = torch.load(BASE_SAE_PATH, map_location="cpu")
ft_ckpt   = torch.load(FT_SAE_PATH, map_location="cpu")

base_dec = base_ckpt["W_dec"].float()
ft_dec   = ft_ckpt["W_dec"].float()

print("Base decoder:", base_dec.shape)
print("FT decoder:", ft_dec.shape)

# --------------------------------------------------
# Normalize decoder vectors
# --------------------------------------------------

base_dec = base_dec / (
    torch.norm(base_dec, dim=1, keepdim=True) + 1e-8
)

ft_dec = ft_dec / (
    torch.norm(ft_dec, dim=1, keepdim=True) + 1e-8
)

# --------------------------------------------------
# Cosine similarity matrix
# --------------------------------------------------

print("\nComputing cosine similarity matrix...")

cos_matrix = base_dec @ ft_dec.T

print("Matrix shape:", cos_matrix.shape)

# --------------------------------------------------
# Best match for each base feature
# --------------------------------------------------

best_cosines, best_indices = torch.max(
    cos_matrix,
    dim=1
)

best_cosines_np = best_cosines.numpy()

# --------------------------------------------------
# Summary stats
# --------------------------------------------------

print("\n==============================")
print("MATCHED FEATURE STATISTICS")
print("==============================")

print(f"Mean   : {best_cosines_np.mean():.6f}")
print(f"Median : {np.median(best_cosines_np):.6f}")
print(f"Std    : {best_cosines_np.std():.6f}")
print(f"Min    : {best_cosines_np.min():.6f}")
print(f"Max    : {best_cosines_np.max():.6f}")

# --------------------------------------------------
# Threshold counts
# --------------------------------------------------

thresholds = [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
]

print("\n==============================")
print("PRESERVATION COUNTS")
print("==============================")

for t in thresholds:
    pct = (best_cosines_np > t).mean() * 100
    print(f">{t:.1f}: {pct:.2f}%")

# --------------------------------------------------
# Top preserved features
# --------------------------------------------------

print("\n==============================")
print("TOP 20 PRESERVED FEATURES")
print("==============================")

top_idx = torch.argsort(
    best_cosines,
    descending=True
)

for rank in range(20):
    i = top_idx[rank].item()

    print(
        f"{rank+1:2d} | "
        f"Base Feature {i:4d} "
        f"-> FT Feature {best_indices[i].item():4d} "
        f"| Cos = {best_cosines[i].item():.6f}"
    )

# --------------------------------------------------
# Most changed features
# --------------------------------------------------

print("\n==============================")
print("TOP 20 MOST CHANGED FEATURES")
print("==============================")

bottom_idx = torch.argsort(
    best_cosines,
    descending=False
)

for rank in range(20):
    i = bottom_idx[rank].item()

    print(
        f"{rank+1:2d} | "
        f"Base Feature {i:4d} "
        f"-> FT Feature {best_indices[i].item():4d} "
        f"| Cos = {best_cosines[i].item():.6f}"
    )

# --------------------------------------------------
# Save results
# --------------------------------------------------

torch.save(
    {
        "best_cosines": best_cosines,
        "best_indices": best_indices,
    },
    "feature_matching_results.pt"
)

print("\nSaved:")
print("feature_matching_results.pt")

Loading checkpoints...
Base decoder: torch.Size([6144, 768])
FT decoder: torch.Size([6144, 768])

Computing cosine similarity matrix...
Matrix shape: torch.Size([6144, 6144])

MATCHED FEATURE STATISTICS
Mean   : 0.134678
Median : 0.133261
Std    : 0.011308
Min    : 0.108582
Max    : 0.209502

PRESERVATION COUNTS
>0.1: 100.00%
>0.2: 0.05%
>0.3: 0.00%
>0.4: 0.00%
>0.5: 0.00%
>0.6: 0.00%
>0.7: 0.00%
>0.8: 0.00%

TOP 20 PRESERVED FEATURES
 1 | Base Feature  683 -> FT Feature 4443 | Cos = 0.209502
 2 | Base Feature 1756 -> FT Feature 1984 | Cos = 0.204054
 3 | Base Feature 5508 -> FT Feature  678 | Cos = 0.201365
 4 | Base Feature 1206 -> FT Feature 3632 | Cos = 0.197624
 5 | Base Feature 4878 -> FT Feature 3158 | Cos = 0.194939
 6 | Base Feature 1287 -> FT Feature 3151 | Cos = 0.189681
 7 | Base Feature 5610 -> FT Feature 5406 | Cos = 0.187936
 8 | Base Feature 2969 -> FT Feature 3153 | Cos = 0.185980
 9 | Base Feature 2762 -> FT Feature 2648 | Cos = 0.185640
10 | Base Feature 5502 -> FT F

In [ ]:
import torch

results = torch.load("feature_matching_results.pt")

best_indices = results["best_indices"]

unique_matches = len(torch.unique(best_indices))

print("Unique FT features matched:", unique_matches)
print("Total base features:", len(best_indices))

Unique FT features matched: 3882
Total base features: 6144


In [ ]:
from sae_lens import StandardSAEConfig, StandardSAE
import torch

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device="cpu",
)

sae = StandardSAE(cfg)

print(hasattr(sae, "encode"))
print(hasattr(sae, "decode"))

print([x for x in dir(sae) if "encode" in x.lower()])

True
True
['encode']


In [ ]:
import torch
import numpy as np
from sae_lens import StandardSAE, StandardSAEConfig

# ============================================================
# PATHS
# ============================================================

BASE_ACT_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/activations/base/layer6_resid_post (1).pt"

FT_ACT_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/activations/finetuned/layer6_resid_post_finetuned.pt"

BASE_SAE_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/base/base_sae_final.pt"

FT_SAE_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/finetuned/finetuned_sae_final.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# LOAD SAES
# ============================================================

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device=device,
)

base_sae = StandardSAE(cfg).to(device)
ft_sae   = StandardSAE(cfg).to(device)

base_sae.load_state_dict(
    torch.load(BASE_SAE_PATH, map_location=device)
)

ft_sae.load_state_dict(
    torch.load(FT_SAE_PATH, map_location=device)
)

base_sae.eval()
ft_sae.eval()

# ============================================================
# LOAD ACTIVATIONS
# ============================================================

base_act = torch.load(BASE_ACT_PATH, map_location="cpu")
ft_act   = torch.load(FT_ACT_PATH, map_location="cpu")

print("Base act:", base_act.shape)
print("FT act:", ft_act.shape)

# ============================================================
# ANALYSIS FUNCTION
# ============================================================

@torch.no_grad()
def analyze(name, sae, activations, batch_size=4096):

    all_firing = []
    all_l0 = []
    all_mag = []

    n = activations.shape[0]

    for start in range(0, n, batch_size):

        end = min(start + batch_size, n)

        x = activations[start:end].to(device)

        feats = sae.encode(x)

        firing = (feats > 0).float()

        all_firing.append(firing.cpu())

        l0 = firing.sum(dim=1)

        all_l0.append(l0.cpu())

        all_mag.append(feats.mean().cpu())

    firing = torch.cat(all_firing, dim=0)

    firing_rate = firing.mean(dim=0)

    l0 = torch.cat(all_l0)

    mean_firing = firing_rate.mean().item()
    median_firing = firing_rate.median().item()

    mean_l0 = l0.mean().item()
    median_l0 = l0.median().item()

    mean_mag = torch.stack(all_mag).mean().item()

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(f"Mean firing rate     : {mean_firing:.6f}")
    print(f"Median firing rate   : {median_firing:.6f}")
    print(f"Mean active features : {mean_l0:.3f}")
    print(f"Median active feats  : {median_l0:.3f}")
    print(f"Mean activation mag  : {mean_mag:.6f}")

    return {
        "firing_rate": firing_rate,
        "mean_l0": mean_l0,
        "median_l0": median_l0,
        "mean_mag": mean_mag,
    }

# ============================================================
# RUN ALL FOUR COMBINATIONS
# ============================================================

res_bb = analyze(
    "BASE SAE on BASE activations",
    base_sae,
    base_act
)

res_bf = analyze(
    "BASE SAE on FT activations",
    base_sae,
    ft_act
)

res_fb = analyze(
    "FT SAE on BASE activations",
    ft_sae,
    base_act
)

res_ff = analyze(
    "FT SAE on FT activations",
    ft_sae,
    ft_act
)

Base act: torch.Size([139549, 768])
FT act: torch.Size([139549, 768])

BASE SAE on BASE activations
Mean firing rate     : 0.057464
Median firing rate   : 0.016761
Mean active features : 353.059
Median active feats  : 307.000
Mean activation mag  : 0.084539

BASE SAE on FT activations
Mean firing rate     : 0.071407
Median firing rate   : 0.020975
Mean active features : 438.724
Median active feats  : 388.000
Mean activation mag  : 0.093650

FT SAE on BASE activations
Mean firing rate     : 0.092872
Median firing rate   : 0.037134
Mean active features : 570.607
Median active feats  : 512.000
Mean activation mag  : 0.099135

FT SAE on FT activations
Mean firing rate     : 0.059054
Median firing rate   : 0.010047
Mean active features : 362.827
Median active feats  : 315.000
Mean activation mag  : 0.090204


In [ ]:
import torch
from sae_lens import StandardSAE, StandardSAEConfig

# =====================================================
# PATHS
# =====================================================

ACT_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/activations/base/layer6_resid_post (1).pt"

TOKEN_MAP_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/analysis/token_map.pt"

SAE_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/base/base_sae_final.pt"

FEATURE_ID = 683   # change later

TOP_K = 30
BATCH_SIZE = 4096

device = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================
# LOAD SAE
# =====================================================

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device=device,
)

sae = StandardSAE(cfg).to(device)

sae.load_state_dict(
    torch.load(SAE_PATH, map_location=device)
)

sae.eval()

# =====================================================
# LOAD DATA
# =====================================================

activations = torch.load(ACT_PATH, map_location="cpu")
token_map = torch.load(TOKEN_MAP_PATH)

print("Activations:", activations.shape)
print("Token records:", len(token_map))

# =====================================================
# COMPUTE FEATURE ACTIVATIONS
# =====================================================

all_scores = []

with torch.no_grad():

    for start in range(0, len(activations), BATCH_SIZE):

        end = min(start + BATCH_SIZE, len(activations))

        x = activations[start:end].to(device)

        feats = sae.encode(x)

        scores = feats[:, FEATURE_ID]

        all_scores.append(scores.cpu())

scores = torch.cat(all_scores)

print("Scores shape:", scores.shape)

# =====================================================
# TOP ACTIVATIONS
# =====================================================

top_vals, top_idx = torch.topk(
    scores,
    TOP_K
)

print("\n")
print("=" * 80)
print(f"FEATURE {FEATURE_ID}")
print("=" * 80)

for rank in range(TOP_K):

    idx = top_idx[rank].item()

    rec = token_map[idx]

    print("\n")
    print("-" * 80)

    print(
        f"Rank {rank+1:2d} | "
        f"Activation = {top_vals[rank].item():.4f}"
    )

    print("Token:")
    print(repr(rec["token"]))

    print("\nFull Context:")
    print(rec["full_text"][:500])

Activations: torch.Size([139549, 768])
Token records: 139549
Scores shape: torch.Size([139549])


FEATURE 683


--------------------------------------------------------------------------------
Rank  1 | Activation = 7.6308
Token:
'ively'

Full Context:
Lively Numbers ; Macmillan , 1957


--------------------------------------------------------------------------------
Rank  2 | Activation = 7.6308
Token:
'ively'

Full Context:
Lively Words ; Macmillan , 1961 .


--------------------------------------------------------------------------------
Rank  3 | Activation = 7.6308
Token:
'ively'

Full Context:
Lively Stories ; Macmillan , 1954


--------------------------------------------------------------------------------
Rank  4 | Activation = 7.2194
Token:
'uri'

Full Context:
Living in Ramgarh , the jovial Veeru and cynical Jai find themselves growing fond of the villagers . Veeru is attracted to Basanti ( Hema Malini ) , a feisty , talkative young woman who makes her living by driving a ho

In [ ]:
import torch
from tqdm import tqdm
from sae_lens import StandardSAE, StandardSAEConfig

# =====================================================
# PATHS
# =====================================================

ACT_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/activations/finetuned/layer6_resid_post_finetuned.pt"

SAE_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/finetuned/finetuned_sae_final.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================
# LOAD SAE
# =====================================================

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device=device,
)

sae = StandardSAE(cfg).to(device)

sae.load_state_dict(
    torch.load(SAE_PATH, map_location=device)
)

sae.eval()

# =====================================================
# LOAD ACTIVATIONS
# =====================================================

acts = torch.load(ACT_PATH, map_location="cpu")

print("Activations:", acts.shape)

# =====================================================
# ACCUMULATORS
# =====================================================

n_features = 6144

fires = torch.zeros(n_features)
max_vals = torch.zeros(n_features)
mean_vals = torch.zeros(n_features)

count = 0

BATCH = 4096

with torch.no_grad():

    for start in tqdm(range(0, len(acts), BATCH)):

        end = min(start + BATCH, len(acts))

        x = acts[start:end].to(device)

        feats = sae.encode(x)

        fires += (feats > 0).float().sum(0).cpu()

        max_vals = torch.maximum(
            max_vals,
            feats.max(0).values.cpu()
        )

        mean_vals += feats.sum(0).cpu()

        count += feats.shape[0]

firing_rate = fires / count
mean_activation = mean_vals / count

# =====================================================
# FILTER
# =====================================================

mask = (
    (firing_rate > 0.002)
    &
    (firing_rate < 0.10)
    &
    (max_vals > 1.0)
)

candidate_ids = torch.where(mask)[0]

print("\nCandidate features:", len(candidate_ids))

# sort by max activation

sorted_ids = candidate_ids[
    torch.argsort(
        max_vals[candidate_ids],
        descending=True
    )
]

print("\nTOP 50 CANDIDATES\n")

for i in sorted_ids[:50]:

    print(
        f"Feature {i.item():4d} | "
        f"firing={firing_rate[i]:.4f} | "
        f"max={max_vals[i]:.4f} | "
        f"mean={mean_activation[i]:.4f}"
    )

Activations: torch.Size([139549, 768])


100%|██████████| 35/35 [00:00<00:00, 54.41it/s]



Candidate features: 3306

TOP 50 CANDIDATES

Feature 1626 | firing=0.0023 | max=74.1256 | mean=0.1283
Feature 4677 | firing=0.0082 | max=73.5553 | mean=0.1320
Feature 1047 | firing=0.0088 | max=71.6819 | mean=0.1274
Feature 1525 | firing=0.0153 | max=67.9563 | mean=0.1384
Feature   68 | firing=0.0027 | max=65.9995 | mean=0.1148
Feature 4155 | firing=0.0171 | max=63.0699 | mean=0.1352
Feature 1000 | firing=0.0127 | max=62.6812 | mean=0.1158
Feature  573 | firing=0.0221 | max=61.4355 | mean=0.2150
Feature  713 | firing=0.0028 | max=58.9400 | mean=0.1016
Feature 5535 | firing=0.0209 | max=55.5644 | mean=0.1088
Feature 5517 | firing=0.0251 | max=53.3035 | mean=0.3014
Feature  973 | firing=0.0230 | max=53.0570 | mean=0.4171
Feature  311 | firing=0.0022 | max=51.7786 | mean=0.0896
Feature 5018 | firing=0.0182 | max=50.7807 | mean=0.1047
Feature 2621 | firing=0.0096 | max=50.6339 | mean=0.0897
Feature 5636 | firing=0.0313 | max=50.5704 | mean=0.2374
Feature 5958 | firing=0.0256 | max=50.4950

In [ ]:
import torch
from sae_lens import StandardSAE, StandardSAEConfig

# =====================================================
# PATHS
# =====================================================

ACT_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/activations/finetuned/layer6_resid_post_finetuned.pt"

TOKEN_MAP_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/analysis/token_map.pt"

SAE_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/finetuned/finetuned_sae_final.pt"

FEATURE_ID = 1626

TOP_K = 50
BATCH_SIZE = 4096

device = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================
# LOAD SAE
# =====================================================

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device=device,
)

sae = StandardSAE(cfg).to(device)

sae.load_state_dict(
    torch.load(SAE_PATH, map_location=device)
)

sae.eval()

# =====================================================
# LOAD DATA
# =====================================================

activations = torch.load(ACT_PATH, map_location="cpu")
token_map = torch.load(TOKEN_MAP_PATH)

print("Activations:", activations.shape)
print("Token records:", len(token_map))

# =====================================================
# COMPUTE FEATURE ACTIVATIONS
# =====================================================

all_scores = []

with torch.no_grad():

    for start in range(0, len(activations), BATCH_SIZE):

        end = min(start + BATCH_SIZE, len(activations))

        x = activations[start:end].to(device)

        feats = sae.encode(x)

        scores = feats[:, FEATURE_ID]

        all_scores.append(scores.cpu())

scores = torch.cat(all_scores)

print("Scores shape:", scores.shape)

# =====================================================
# TOP ACTIVATIONS
# =====================================================

top_vals, top_idx = torch.topk(
    scores,
    TOP_K
)

print("\n")
print("=" * 100)
print(f"FEATURE {FEATURE_ID}")
print("=" * 100)

for rank in range(TOP_K):

    idx = top_idx[rank].item()

    rec = token_map[idx]

    print("\n")
    print("-" * 100)

    print(
        f"Rank {rank+1:2d} | "
        f"Activation = {top_vals[rank].item():.4f}"
    )

    print(f"Global Index: {idx}")

    print("Token:")
    print(repr(rec["token"]))

    print("\nPosition in text:")
    print(rec["position"])

    print("\nFull Context:")
    print(rec["full_text"][:800])

print("\nDone.")

Activations: torch.Size([139549, 768])
Token records: 139549
Scores shape: torch.Size([139549])


FEATURE 1626


----------------------------------------------------------------------------------------------------
Rank  1 | Activation = 74.1256
Global Index: 74842
Token:
'.'

Position in text:
26

Full Context:
The Great Coastal hurricane of 1806 was first noted far east of the Lesser Antilles on 17 August . Weather historian David M. Ludlum followed the disturbance 's track to the Bahamas by 19 August ; intense winds persisted until 21 August , however , approximately 150 mi ( 240 km ) east of the Bahamian island of Eleuthera . Steering currents brought the storm northward , and it approached Charleston , South Carolina on 22 August , where a generally easterly flow preceded the storm indicated its passage far east of the city . The hurricane made landfall at the mouth of the Cape Fear River in North Carolina later that day , though the earliest impacts from the storm started several 

In [ ]:
import torch
from sae_lens import StandardSAE, StandardSAEConfig

# =====================================================
# PATHS
# =====================================================

ACT_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/activations/finetuned/layer6_resid_post_finetuned.pt"

TOKEN_MAP_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/analysis/token_map.pt"

SAE_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/finetuned/finetuned_sae_final.pt"

FEATURES = [
    68,
    311,
    713,
    1047,
    1626,
    4677,
    6098
]

TOP_K = 10
BATCH_SIZE = 4096

device = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================
# LOAD SAE
# =====================================================

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device=device,
)

sae = StandardSAE(cfg).to(device)

sae.load_state_dict(
    torch.load(SAE_PATH, map_location=device)
)

sae.eval()

# =====================================================
# LOAD DATA
# =====================================================

acts = torch.load(ACT_PATH, map_location="cpu")
token_map = torch.load(TOKEN_MAP_PATH)

# =====================================================
# COMPUTE FEATURE ACTIVATIONS
# =====================================================

with torch.no_grad():

    for feature_id in FEATURES:

        print("\n")
        print("=" * 80)
        print(f"FEATURE {feature_id}")
        print("=" * 80)

        scores_list = []

        for start in range(0, len(acts), BATCH_SIZE):

            end = min(start + BATCH_SIZE, len(acts))

            x = acts[start:end].to(device)

            feats = sae.encode(x)

            scores_list.append(
                feats[:, feature_id].cpu()
            )

        scores = torch.cat(scores_list)

        top_vals, top_idx = torch.topk(
            scores,
            TOP_K
        )

        for rank in range(TOP_K):

            idx = top_idx[rank].item()

            rec = token_map[idx]

            print(
                f"{rank+1:2d}. "
                f"Act={top_vals[rank].item():.2f} | "
                f"Token={repr(rec['token'])}"
            )



FEATURE 68
 1. Act=66.00 | Token='.'
 2. Act=65.89 | Token='.'
 3. Act=65.89 | Token='.'
 4. Act=65.85 | Token='.'
 5. Act=65.84 | Token='.'
 6. Act=65.81 | Token='.'
 7. Act=65.75 | Token='.'
 8. Act=65.75 | Token='.'
 9. Act=65.69 | Token='.'
10. Act=65.67 | Token='.'


FEATURE 311
 1. Act=51.78 | Token='.'
 2. Act=51.77 | Token='.'
 3. Act=51.72 | Token='.'
 4. Act=51.71 | Token='.'
 5. Act=51.64 | Token='.'
 6. Act=51.55 | Token='.'
 7. Act=51.53 | Token='.'
 8. Act=51.53 | Token='.'
 9. Act=51.51 | Token='.'
10. Act=51.45 | Token='.'


FEATURE 713
 1. Act=58.94 | Token='.'
 2. Act=58.76 | Token='.'
 3. Act=58.63 | Token='.'
 4. Act=58.60 | Token='.'
 5. Act=58.58 | Token='.'
 6. Act=58.58 | Token='.'
 7. Act=58.53 | Token='.'
 8. Act=58.37 | Token='.'
 9. Act=58.36 | Token='.'
10. Act=58.34 | Token='.'


FEATURE 1047
 1. Act=71.68 | Token='.'
 2. Act=71.64 | Token='.'
 3. Act=71.51 | Token='.'
 4. Act=71.47 | Token='.'
 5. Act=71.45 | Token='.'
 6. Act=71.30 | Token='.'
 7. Act=

In [ ]:
import torch
from tqdm import tqdm
from sae_lens import StandardSAE, StandardSAEConfig

ACT_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/activations/finetuned/layer6_resid_post_finetuned.pt"
SAE_PATH = "/content/drive/MyDrive/Mechanistic_Interpretability/sae_models/finetuned/finetuned_sae_final.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device=device,
)

sae = StandardSAE(cfg).to(device)
sae.load_state_dict(torch.load(SAE_PATH, map_location=device))
sae.eval()

acts = torch.load(ACT_PATH, map_location="cpu")

fires = torch.zeros(6144)
means = torch.zeros(6144)

BATCH = 4096
count = 0

with torch.no_grad():

    for start in tqdm(range(0, len(acts), BATCH)):

        end = min(start + BATCH, len(acts))

        x = acts[start:end].to(device)

        feats = sae.encode(x)

        fires += (feats > 0).float().sum(0).cpu()
        means += feats.sum(0).cpu()

        count += feats.shape[0]

firing_rate = fires / count
mean_activation = means / count

score = mean_activation * (1.0 - firing_rate)

top = torch.argsort(score, descending=True)

print("\nTOP 100 FEATURES\n")

for i in top[:100]:

    print(
        f"Feature {i.item():4d} | "
        f"firing={firing_rate[i]:.4f} | "
        f"mean={mean_activation[i]:.4f}"
    )

100%|██████████| 35/35 [00:00<00:00, 58.31it/s]


TOP 100 FEATURES

Feature 4223 | firing=0.0281 | mean=0.6502
Feature  533 | firing=0.0263 | mean=0.6437
Feature 2501 | firing=0.0283 | mean=0.6397
Feature 1014 | firing=0.0287 | mean=0.6379
Feature 5132 | firing=0.0282 | mean=0.6227
Feature 4259 | firing=0.0288 | mean=0.6221
Feature 5508 | firing=0.0278 | mean=0.6078
Feature 4905 | firing=0.0269 | mean=0.6028
Feature 1112 | firing=0.0259 | mean=0.6017
Feature 3622 | firing=0.0288 | mean=0.6033
Feature 3071 | firing=0.0263 | mean=0.6013
Feature 4232 | firing=0.0253 | mean=0.5997
Feature 1646 | firing=0.0358 | mean=0.6038
Feature 3101 | firing=0.0326 | mean=0.5903
Feature 4965 | firing=0.0269 | mean=0.5794
Feature 6027 | firing=0.0330 | mean=0.5822
Feature 4212 | firing=0.0295 | mean=0.5789
Feature 3757 | firing=0.0333 | mean=0.5808
Feature 2478 | firing=0.0290 | mean=0.5759
Feature 4974 | firing=0.0351 | mean=0.5684
Feature 4200 | firing=0.0308 | mean=0.5573
Feature 2285 | firing=0.0282 | mean=0.5512
Feature  633 | firing=0.0328 | mean

In [ ]:
import torch

token_map = torch.load(
    "/content/drive/MyDrive/Mechanistic_Interpretability/analysis/token_map.pt"
)

print(type(token_map))
print(len(token_map))

print("\nFirst entry:\n")
print(token_map[0])

<class 'list'>
139549

First entry:

{'global_idx': 0, 'token': '=', 'full_text': '= Valkyria Chronicles III =', 'position': 0}


In [5]:
import torch

ACT_PATH = "/content/drive/MyDrive/MechInterp/activations/base/layer6_resid_post.pt"

acts = torch.load(
    ACT_PATH,
    map_location="cpu"
)

print(acts.shape)
print("Mean:", acts.mean().item())
print("Std :", acts.std().item())

torch.Size([255984, 768])
Mean: -0.012337015941739082
Std : 1.2052631378173828


In [8]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sae_lens import StandardSAE, StandardSAEConfig
from tqdm import tqdm

device = "cuda"

# ====================================================
# LOAD ACTIVATIONS
# ====================================================

ACT_PATH = "/content/drive/MyDrive/MechInterp/activations/base/layer6_resid_post.pt"

acts = torch.load(
    ACT_PATH,
    map_location="cpu"
)

print("Activations:", acts.shape)

dataset = TensorDataset(acts)

loader = DataLoader(
    dataset,
    batch_size=1024,
    shuffle=True,
    drop_last=True
)

# ====================================================
# SAE
# ====================================================

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device=device,
)

sae = StandardSAE(cfg).to(device)

optimizer = torch.optim.Adam(
    sae.parameters(),
    lr=1e-4
)

L1_COEF = 1e-3

EPOCHS = 10

# ====================================================
# TRAIN
# ====================================================

for epoch in range(EPOCHS):

    sae.train()

    running_loss = 0.0

    pbar = tqdm(loader)

    for batch in pbar:

        x = batch[0].to(device)

        features = sae.encode(x)

        recon = sae.decode(features)

        recon_loss = ((recon - x) ** 2).mean()

        l1_loss = features.abs().mean()

        loss = recon_loss + L1_COEF * l1_loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        pbar.set_description(
            f"Epoch {epoch+1} "
            f"Loss={loss.item():.5f} "
            f"Recon={recon_loss.item():.5f}"
        )

    avg_loss = running_loss / len(loader)

    print(
        f"\nEpoch {epoch+1} "
        f"Avg Loss = {avg_loss:.6f}"
    )

Activations: torch.Size([255984, 768])


Epoch 1 Loss=1.90331 Recon=1.90325: 100%|██████████| 249/249 [00:07<00:00, 32.29it/s]



Epoch 1 Avg Loss = 20.009271


Epoch 2 Loss=0.33607 Recon=0.33602: 100%|██████████| 249/249 [00:06<00:00, 39.83it/s]



Epoch 2 Avg Loss = 0.809973


...


Epoch 8 Avg Loss = 0.120917


Epoch 9 Loss=0.10181 Recon=0.10176: 100%|██████████| 249/249 [00:06<00:00, 40.83it/s]



Epoch 9 Avg Loss = 0.105821


Epoch 10 Loss=0.08794 Recon=0.08790: 100%|██████████| 249/249 [00:06<00:00, 40.44it/s]


Epoch 10 Avg Loss = 0.093837


In [9]:
import torch

sae.eval()

sample = acts[:5000].to(device)

with torch.no_grad():

    feats = sae.encode(sample)

    recon = sae.decode(feats)

    recon_loss = (
        (recon - sample) ** 2
    ).mean()

    active = (
        feats > 0
    ).sum(dim=1)

print("Recon:", recon_loss.item())

print(
    "Mean active:",
    active.float().mean().item()
)

print(
    "Median active:",
    active.float().median().item()
)

Recon: 0.09438637644052505
Mean active: 320.591796875
Median active: 288.0


In [10]:
SAVE_PATH = "/content/drive/MyDrive/MechInterp/sae_models/base/base_sae_final.pt"

torch.save(
    sae.state_dict(),
    SAVE_PATH
)

print("Saved:", SAVE_PATH)

Saved: /content/drive/MyDrive/MechInterp/sae_models/base/base_sae_final.pt


In [11]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sae_lens import StandardSAE, StandardSAEConfig
from tqdm import tqdm

device = "cuda"

# ====================================================
# LOAD ACTIVATIONS
# ====================================================

ACT_PATH = "/content/drive/MyDrive/MechInterp/activations/finetuned/layer6_resid_post_finetuned.pt"

acts = torch.load(
    ACT_PATH,
    map_location="cpu"
)

print("Activations:", acts.shape)

dataset = TensorDataset(acts)

loader = DataLoader(
    dataset,
    batch_size=1024,
    shuffle=True,
    drop_last=True
)

# ====================================================
# SAE
# ====================================================

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device=device,
)

sae = StandardSAE(cfg).to(device)

optimizer = torch.optim.Adam(
    sae.parameters(),
    lr=1e-4
)

L1_COEF = 1e-3

EPOCHS = 10

# ====================================================
# TRAIN
# ====================================================

for epoch in range(EPOCHS):

    sae.train()

    running_loss = 0.0

    pbar = tqdm(loader)

    for batch in pbar:

        x = batch[0].to(device)

        features = sae.encode(x)

        recon = sae.decode(features)

        recon_loss = (
            (recon - x) ** 2
        ).mean()

        l1_loss = features.abs().mean()

        loss = (
            recon_loss
            + L1_COEF * l1_loss
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        pbar.set_description(
            f"Epoch {epoch+1} "
            f"Loss={loss.item():.5f} "
            f"Recon={recon_loss.item():.5f}"
        )

    avg_loss = running_loss / len(loader)

    print(
        f"\nEpoch {epoch+1} "
        f"Avg Loss = {avg_loss:.6f}"
    )

Activations: torch.Size([255984, 768])


Epoch 1 Loss=2.05852 Recon=2.05844: 100%|██████████| 249/249 [00:07<00:00, 32.43it/s]



Epoch 1 Avg Loss = 18.675132


Epoch 2 Loss=0.31258 Recon=0.31253: 100%|██████████| 249/249 [00:06<00:00, 38.43it/s]



Epoch 2 Avg Loss = 0.673234


...


Epoch 8 Avg Loss = 0.121858


Epoch 9 Loss=0.09689 Recon=0.09685: 100%|██████████| 249/249 [00:06<00:00, 37.59it/s]



Epoch 9 Avg Loss = 0.105481


Epoch 10 Loss=0.09094 Recon=0.09087: 100%|██████████| 249/249 [00:06<00:00, 39.99it/s]


Epoch 10 Avg Loss = 0.092021


In [12]:
sae.eval()

sample = acts[:10000].to(device)

with torch.no_grad():

    feats = sae.encode(sample)

    recon = sae.decode(feats)

    recon_loss = (
        (recon - sample) ** 2
    ).mean()

    active = (
        feats > 0
    ).sum(dim=1)

print("Recon:", recon_loss.item())

print(
    "Mean active:",
    active.float().mean().item()
)

print(
    "Median active:",
    active.float().median().item()
)

Recon: 0.08969027549028397
Mean active: 346.123779296875
Median active: 318.0


In [13]:
SAVE_PATH = "/content/drive/MyDrive/MechInterp/sae_models/finetuned/finetuned_sae_final.pt"

torch.save(
    sae.state_dict(),
    SAVE_PATH
)

print("Saved:", SAVE_PATH)

Saved: /content/drive/MyDrive/MechInterp/sae_models/finetuned/finetuned_sae_final.pt


In [14]:
import torch
from sae_lens import StandardSAE, StandardSAEConfig

device = "cuda"

# =====================================================
# PATHS
# =====================================================

BASE_ACTS = "/content/drive/MyDrive/MechInterp/activations/base/layer6_resid_post.pt"

FT_ACTS = "/content/drive/MyDrive/MechInterp/activations/finetuned/layer6_resid_post_finetuned.pt"

BASE_SAE = "/content/drive/MyDrive/MechInterp/sae_models/base/base_sae_final.pt"

FT_SAE = "/content/drive/MyDrive/MechInterp/sae_models/finetuned/finetuned_sae_final.pt"

# =====================================================
# LOAD ACTIVATIONS
# =====================================================

print("Loading activations...")

base_acts = torch.load(BASE_ACTS, map_location="cpu")
ft_acts   = torch.load(FT_ACTS, map_location="cpu")

print("Base:", base_acts.shape)
print("FT  :", ft_acts.shape)

# =====================================================
# LOAD SAES
# =====================================================

cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device=device,
)

base_sae = StandardSAE(cfg).to(device)
ft_sae   = StandardSAE(cfg).to(device)

base_sae.load_state_dict(
    torch.load(BASE_SAE, map_location=device)
)

ft_sae.load_state_dict(
    torch.load(FT_SAE, map_location=device)
)

base_sae.eval()
ft_sae.eval()

# =====================================================
# EVAL FUNCTION
# =====================================================

def reconstruction_loss(sae, acts, batch_size=4096):

    losses = []

    with torch.no_grad():

        for start in range(
            0,
            len(acts),
            batch_size
        ):

            batch = acts[
                start:start+batch_size
            ].to(device)

            feats = sae.encode(batch)

            recon = sae.decode(feats)

            loss = (
                (recon - batch) ** 2
            ).mean()

            losses.append(
                loss.item()
            )

    return sum(losses) / len(losses)

# =====================================================
# TRANSFER MATRIX
# =====================================================

print("\nRunning evaluations...\n")

bb = reconstruction_loss(
    base_sae,
    base_acts
)

bf = reconstruction_loss(
    base_sae,
    ft_acts
)

fb = reconstruction_loss(
    ft_sae,
    base_acts
)

ff = reconstruction_loss(
    ft_sae,
    ft_acts
)

print("=" * 60)
print("RECONSTRUCTION TRANSFER MATRIX")
print("=" * 60)

print(f"\nBase SAE on Base Activations : {bb:.6f}")
print(f"Base SAE on FT Activations   : {bf:.6f}")

print(f"\nFT SAE on Base Activations   : {fb:.6f}")
print(f"FT SAE on FT Activations     : {ff:.6f}")

Loading activations...
Base: torch.Size([255984, 768])
FT  : torch.Size([255984, 768])

Running evaluations...

RECONSTRUCTION TRANSFER MATRIX

Base SAE on Base Activations : 0.088149
Base SAE on FT Activations   : 0.257029

FT SAE on Base Activations   : 0.354776
FT SAE on FT Activations     : 0.086003


In [15]:
import torch

base = torch.load(
    "/content/drive/MyDrive/MechInterp/activations/base/layer6_resid_post.pt",
    map_location="cpu"
)

ft = torch.load(
    "/content/drive/MyDrive/MechInterp/activations/finetuned/layer6_resid_post_finetuned.pt",
    map_location="cpu"
)

drift = (ft - base).norm(dim=1)

print("Mean drift :", drift.mean().item())
print("Median drift:", drift.median().item())
print("Max drift :", drift.max().item())

Mean drift : 8.931478500366211
Median drift: 7.951946258544922
Max drift : 100.0769271850586


In [16]:
topk = torch.topk(drift, k=1000)

torch.save(
    topk.indices,
    "/content/drive/MyDrive/MechInterp/analysis/top_drift_indices.pt"
)

In [17]:
import torch

base = torch.load(
    "/content/drive/MyDrive/MechInterp/activations/base/layer6_resid_post.pt",
    map_location="cpu"
)

ft = torch.load(
    "/content/drive/MyDrive/MechInterp/activations/finetuned/layer6_resid_post_finetuned.pt",
    map_location="cpu"
)

token_map = torch.load(
    "/content/drive/MyDrive/MechInterp/analysis/token_map.pt"
)

drift = (ft - base).norm(dim=1)

topk = torch.topk(drift, k=100)

print("\nTOP 100 DRIFT TOKENS\n")

for rank, idx in enumerate(topk.indices[:100]):

    rec = token_map[int(idx)]

    print(
        f"{rank+1:3d} | "
        f"Drift={drift[idx]:.3f} | "
        f"Token={repr(rec['token'])}"
    )


TOP 100 DRIFT TOKENS

  1 | Drift=100.077 | Token='##'
  2 | Drift=100.077 | Token='##'
  3 | Drift=100.077 | Token='##'
  4 | Drift=100.077 | Token='##'
  5 | Drift=90.224 | Token=' permitted'
  6 | Drift=75.502 | Token=' permitted'
  7 | Drift=63.647 | Token=' The'
  8 | Drift=63.647 | Token=' The'
  9 | Drift=61.369 | Token='#'
 10 | Drift=61.369 | Token='#'
 11 | Drift=61.369 | Token='#'
 12 | Drift=61.369 | Token='#'
 13 | Drift=61.369 | Token='#'
 14 | Drift=61.369 | Token='#'
 15 | Drift=61.369 | Token='#'
 16 | Drift=61.369 | Token='#'
 17 | Drift=61.369 | Token='#'
 18 | Drift=61.369 | Token='#'
 19 | Drift=61.369 | Token='#'
 20 | Drift=61.369 | Token='#'
 21 | Drift=61.369 | Token='#'
 22 | Drift=61.369 | Token='#'
 23 | Drift=61.369 | Token='#'
 24 | Drift=61.369 | Token='#'
 25 | Drift=61.369 | Token='#'
 26 | Drift=61.369 | Token='#'
 27 | Drift=61.369 | Token='#'
 28 | Drift=61.369 | Token='#'
 29 | Drift=61.369 | Token='#'
 30 | Drift=61.369 | Token='#'
 31 | Drift=61.

In [18]:
from collections import Counter
import torch

token_map = torch.load(
    "/content/drive/MyDrive/MechInterp/analysis/token_map.pt"
)

drift = (ft - base).norm(dim=1)

topk = torch.topk(drift, k=5000)

counter = Counter()

for idx in topk.indices:

    token = token_map[int(idx)]["token"]

    counter[token] += 1

print("\nMOST COMMON HIGH-DRIFT TOKENS\n")

for tok, count in counter.most_common(50):
    print(repr(tok), count)


MOST COMMON HIGH-DRIFT TOKENS

'#' 632
'\n' 220
'_' 210
'.' 188
'self' 167
'"""' 138
'from' 138
'import' 85
',' 80
' self' 69
'(' 47
'1' 47
')' 45
"'," 40
' =' 38
'2' 37
'name' 33
' The' 31
'    ' 29
':' 28
"'" 26
'        ' 25
'-' 22
's' 21
'Error' 21
'def' 20
'True' 19
'0' 18
'path' 17
'y' 17
' 0' 16
'Field' 16
'\n\n' 16
'paces' 15
' 2' 15
'()' 15
'3' 15
'b' 14
'"' 14
'html' 14
'com' 14
' None' 13
'max' 13
" '" 13
' x' 12
'p' 12
'assertEqual' 12
']' 12
' (' 12
'add' 12


In [19]:
from collections import defaultdict
import torch

base = torch.load(
    "/content/drive/MyDrive/MechInterp/activations/base/layer6_resid_post.pt",
    map_location="cpu"
)

ft = torch.load(
    "/content/drive/MyDrive/MechInterp/activations/finetuned/layer6_resid_post_finetuned.pt",
    map_location="cpu"
)

token_map = torch.load(
    "/content/drive/MyDrive/MechInterp/analysis/token_map.pt"
)

drift = (ft - base).norm(dim=1)

token_sum = defaultdict(float)
token_count = defaultdict(int)

for i in range(len(drift)):

    tok = token_map[i]["token"]

    token_sum[tok] += float(drift[i])
    token_count[tok] += 1

avg_drift = []

for tok in token_sum:

    if token_count[tok] >= 20:   # ignore rare tokens

        avg_drift.append((
            token_sum[tok] / token_count[tok],
            token_count[tok],
            tok
        ))

avg_drift.sort(reverse=True)

print("\nTOP TOKENS BY AVERAGE DRIFT\n")

for mean_drift, count, tok in avg_drift[:100]:

    print(
        f"{mean_drift:.3f} | "
        f"count={count:5d} | "
        f"{repr(tok)}"
    )


TOP TOKENS BY AVERAGE DRIFT

18.290 | count=   22 | 'paces'
15.971 | count=   25 | ' permitted'
15.848 | count=  286 | ' self'
14.968 | count=   28 | 'pan'
14.474 | count= 1396 | 'self'
14.472 | count=   21 | 'ses'
14.367 | count=  172 | ' The'
14.155 | count=  125 | 'True'
13.826 | count=   59 | 'append'
13.793 | count=   64 | 'assertEqual'
13.439 | count=   28 | 'components'
13.348 | count=   48 | 'mp'
13.332 | count=   36 | ' 4'
13.229 | count=   37 | 'node'
13.143 | count= 7155 | '#'
13.136 | count=   31 | 'status'
13.099 | count=   99 | 'value'
13.098 | count=   20 | 'permission'
13.074 | count=   20 | 'num'
13.019 | count=   26 | 'decode'
13.008 | count=   21 | 'equal'
12.954 | count=   22 | 'ness'
12.953 | count=   20 | 'foo'
12.943 | count=   29 | '</'
12.849 | count=   39 | ' y'
12.828 | count=   39 | 'item'
12.809 | count=   25 | 'ascii'
12.767 | count=   52 | 'Input'
12.745 | count=   22 | ' transform'
12.738 | count=   23 | 'va'
12.713 | count=   30 | 'exit'
12.646 | count

In [20]:
import torch
from sae_lens import StandardSAE, StandardSAEConfig

device = "cuda"

# load activations
base_acts = torch.load(BASE_ACTS, map_location="cpu")[:50000]
ft_acts   = torch.load(FT_ACTS, map_location="cpu")[:50000]

# load SAEs
cfg = StandardSAEConfig(
    d_in=768,
    d_sae=6144,
    device=device,
)

base_sae = StandardSAE(cfg).to(device)
ft_sae   = StandardSAE(cfg).to(device)

base_sae.load_state_dict(
    torch.load(BASE_SAE, map_location=device)
)

ft_sae.load_state_dict(
    torch.load(FT_SAE, map_location=device)
)

base_sae.eval()
ft_sae.eval()

with torch.no_grad():

    base_feats = base_sae.encode(
        base_acts.to(device)
    )

    ft_feats = ft_sae.encode(
        ft_acts.to(device)
    )

# average activation of each feature

base_mean = base_feats.mean(0).cpu()
ft_mean   = ft_feats.mean(0).cpu()

feature_drift = (ft_mean - base_mean).abs()

topk = torch.topk(
    feature_drift,
    k=50
)

print("\nTOP DRIFTING SAE FEATURES\n")

for rank, idx in enumerate(topk.indices):

    print(
        rank + 1,
        int(idx),
        float(feature_drift[idx])
    )


TOP DRIFTING SAE FEATURES

1 647 15.931365966796875
2 684 10.92683219909668
3 1711 10.49072551727295
4 4415 10.207764625549316
5 3725 6.696202754974365
6 455 6.4067277908325195
7 4074 5.90568733215332
8 3550 5.317445755004883
9 2101 3.98126482963562
10 2772 3.552886724472046
11 3595 3.090040922164917
12 1191 3.0390098094940186
13 2212 3.0015649795532227
14 119 2.967898368835449
15 1479 2.8596720695495605
16 2420 2.807321071624756
17 5525 2.757789373397827
18 4529 2.741358995437622
19 992 2.730147361755371
20 4049 2.610180377960205
21 92 2.5323503017425537
22 764 2.4953415393829346
23 5362 2.437131404876709
24 1293 2.389451026916504
25 179 2.357344150543213
26 1518 2.3273398876190186
27 1365 2.325242280960083
28 1720 2.110886812210083
29 106 1.9904357194900513
30 5416 1.9750910997390747
31 2161 1.9156256914138794
32 6089 1.8940997123718262
33 1460 1.8830386400222778
34 2564 1.8605936765670776
35 4634 1.8458837270736694
36 475 1.8201892375946045
37 4130 1.7751046419143677
38 4409 1.7663